# Sequence Classification

Multi-target attribution on BERT for sentence-level classification.
Each section covers one **input pattern** organised by what the explainer requires.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [25]:
import time

import torch
from transformers import BertForSequenceClassification, BertTokenizerFast

from torchxai.data_types import SingleTargetAcrossBatch

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).eval().cuda()

text = "This movie was absolutely fantastic, but the subplot felt unexpectedly overdramatized."
enc = tokenizer(text, return_tensors="pt")
input_ids = enc["input_ids"].cuda()  # (1, seq_len)
attention_mask = enc["attention_mask"].cuda()  # (1, seq_len)

# Embeddings — used as input for all explainers
embeddings = model.bert.embeddings(input_ids)  # (1, seq_len, 768)
baseline_emb = torch.zeros_like(embeddings)


class EmbeddingSeqCls(torch.nn.Module):
    def forward(self, emb):
        return model(inputs_embeds=emb, attention_mask=attention_mask).logits


embed_model = EmbeddingSeqCls().eval().cuda()

# 2 output classes (positive / negative)
targets = [SingleTargetAcrossBatch(index=i) for i in range(2)]


def compare(explainer_cls, model, explain_kwargs, targets, atol=1e-5, **init_kwargs):
    """Compare sequential single-target calls vs one multi-target call.
    Verifies results match and reports timing and attribution shape.
    Pass atol>1e-5 for stochastic methods (e.g. GradientShap).
    """
    explainer = explainer_cls(model, multi_target=False, **init_kwargs)
    t0 = time.perf_counter()
    attrs = [explainer.explain(**explain_kwargs, target=t) for t in targets]
    elapsed_single = time.perf_counter() - t0
    attrs_tensor = torch.stack(attrs)

    explainer_mt = explainer_cls(model, multi_target=True, **init_kwargs)
    t0 = time.perf_counter()
    attrs_mt = explainer_mt.explain(**explain_kwargs, target=targets)
    elapsed_mt = time.perf_counter() - t0
    attrs_mt_tensor = torch.stack(attrs_mt)

    assert torch.allclose(attrs_tensor, attrs_mt_tensor, atol=atol), (
        f"Results differ between single-target and multi-target, max diff: {(attrs_tensor - attrs_mt_tensor).abs().max().item():.3e}"
    )
    speedup = elapsed_single / elapsed_mt if elapsed_mt > 0 else float("inf")
    print(f"shape  : {attrs_mt_tensor.shape}")
    print(
        f"single : {elapsed_single:.3f}s  |  multi : {elapsed_mt:.3f}s  |  speedup : {speedup:.1f}x"
    )
    return attrs_mt_tensor

/mnt/noel/phd-2026/projects/torchxai/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


---
## Pattern A — inputs + target

No baseline or mask required.

Applies to: `SaliencyExplainer`, `InputXGradientExplainer`, `GuidedBackpropExplainer`, `RandomExplainer`, `FeatureAblationExplainer`, `LimeExplainer`, `KernelShapExplainer`.

In [32]:
from torchxai.explainers import SaliencyExplainer

# Replace with InputXGradientExplainer as needed

result = compare(SaliencyExplainer, embed_model, {"inputs": embeddings}, targets)
print(f"Output shape={result.shape}, mean={result.mean().item():.3f}, std={result.std().item():.3f}, min={result.min().item():.3f}, max={result.max().item():.3f}")

shape  : torch.Size([2, 1, 20, 768])
single : 0.011s  |  multi : 0.011s  |  speedup : 1.0x
Output shape=torch.Size([2, 1, 20, 768]), mean=-0.000, std=0.000, min=-0.003, max=0.003


---
## Pattern B — inputs + baseline + target

A single reference tensor (same shape as `inputs`) is required.

Applies to: `IntegratedGradientsExplainer`, `DeepLiftExplainer`, `InputXBaselineGradientExplainer`.

In [31]:
from torchxai.explainers import IntegratedGradientsExplainer

# Replace with InputXBaselineGradientExplainer as needed

result = compare(IntegratedGradientsExplainer, embed_model,
        {"inputs": embeddings, "baselines": baseline_emb}, targets)
print(f"Output shape={result.shape}, mean={result.mean().item():.3f}, std={result.std().item():.3f}, min={result.min().item():.3f}, max={result.max().item():.3f}")

shape  : torch.Size([2, 1, 20, 768])
single : 0.150s  |  multi : 0.131s  |  speedup : 1.2x
Result tensor: torch.Size([2, 1, 20, 768])
Output shape=torch.Size([2, 1, 20, 768]), mean=0.000, std=0.001, min=-0.058, max=0.014


---
## Pattern C — inputs + baseline distribution + target

`baselines` is a **stacked set of reference samples** rather than a single tensor.

Applies to: `DeepLiftShapExplainer`, `GradientShapExplainer`.

In [30]:
from torchxai.explainers import GradientShapExplainer

baselines_dist = baseline_emb.expand(5, -1, -1)   # (5, seq_len, 768) reference distribution

# gradient shap has internal non-determinism due to random sampling, so we set a higher tolerance for comparison and higher
# number of samples for better convergence. Adjust as needed for other stochastic explainers.
result = compare(GradientShapExplainer, embed_model,
        {"inputs": embeddings, "baselines": baselines_dist}, targets, n_samples=200, atol=0.1)
print(f"Output shape={result.shape}, mean={result.mean().item():.3f}, std={result.std().item():.3f}, min={result.min().item():.3f}, max={result.max().item():.3f}")

shape  : torch.Size([2, 1, 20, 768])
single : 2.391s  |  multi : 1.629s  |  speedup : 1.5x
Output shape=torch.Size([2, 1, 20, 768]), mean=0.000, std=0.001, min=-0.073, max=0.017


---
## Pattern D — inputs + feature_mask + target

A `feature_mask` groups input elements into segments so the explainer scores whole regions rather than individual pixels.

Applies to: `FeatureAblationExplainer`, `LimeExplainer`, `KernelShapExplainer`.

The same explainer class works with or without a mask. Pixel-level attribution uses no mask; segment-level attribution passes one.

In [35]:
from torchxai.explainers import FeatureAblationExplainer

# Replace with LimeExplainer or KernelShapExplainer as needed

# Word-level feature mask: subword tokens of the same word share an ID
word_ids     = enc.word_ids(batch_index=0)   # e.g. [None, 0, 1, 1, 2, None]
seq_len      = input_ids.shape[1]

ids = input_ids[0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

# Shape (1, seq_len, 1): broadcast across 768 embedding dims
feature_mask = torch.zeros(1, seq_len, 1, dtype=torch.long, device=input_ids.device)
for t_idx, w_idx in enumerate(word_ids):
    if w_idx is not None:
        feature_mask[0, t_idx, 0] = w_idx + 1  # 0 reserved for special tokens

print("\nToken debug view:")
print(f"{'idx':>3}  {'id':>6}  {'word_id':>7}  {'fmask':>5}  token")
print("-" * 46)
feature_mask_ids = feature_mask[0, :, 0].tolist()
for i, (tid, tok, wid, fmask) in enumerate(
    zip(ids, tokens, word_ids, feature_mask_ids, strict=True)
):
    print(f"{i:>3}  {tid:>6}  {str(wid):>7}  {fmask:>5}  {tok}")

print("\nWith word-level feature mask:")
result = compare(FeatureAblationExplainer, embed_model,
        {"inputs": embeddings, "feature_mask": feature_mask}, targets)
print(f"Output shape={result.shape}, mean={result.mean().item():.3f}, std={result.std().item():.3f}, min={result.min().item():.3f}, max={result.max().item():.3f}")


Token debug view:
idx      id  word_id  fmask  token
----------------------------------------------
  0     101     None      0  [CLS]
  1    2023        0      1  this
  2    3185        1      2  movie
  3    2001        2      3  was
  4    7078        3      4  absolutely
  5   10392        4      5  fantastic
  6    1010        5      6  ,
  7    2021        6      7  but
  8    1996        7      8  the
  9    4942        8      9  sub
 10   24759        8      9  ##pl
 11    4140        8      9  ##ot
 12    2371        9     10  felt
 13   14153       10     11  unexpectedly
 14    2058       11     12  over
 15    7265       11     12  ##dra
 16   18900       11     12  ##mat
 17    3550       11     12  ##ized
 18    1012       12     13  .
 19     102     None      0  [SEP]

With word-level feature mask:
shape  : torch.Size([2, 1, 20, 768])
single : 0.020s  |  multi : 0.013s  |  speedup : 1.5x
Output shape=torch.Size([2, 1, 20, 768]), mean=0.002, std=0.028, min=-0.045, max=

---
## Pattern E — inputs + sliding_window_shapes + target

`OcclusionExplainer` patches out rectangular windows instead of using a feature mask.
The `sliding_window_shapes` tuple specifies the window size per input channel.

In [52]:
from torchxai.explainers import OcclusionExplainer

print("\nWith word-level feature mask:")
print(embeddings.shape)
result = compare(OcclusionExplainer, embed_model,
        {"inputs": embeddings, "sliding_window_shapes": (4, 768), "strides": (2, 768)}, targets)
print(f"Output shape={result.shape}, mean={result.mean().item():.3f}, std={result.std().item():.3f}, min={result.min().item():.3f}, max={result.max().item():.3f}")


With word-level feature mask:
torch.Size([1, 20, 768])
shape  : torch.Size([2, 1, 20, 768])
single : 0.046s  |  multi : 0.028s  |  speedup : 1.7x
Output shape=torch.Size([2, 1, 20, 768]), mean=0.000, std=0.000, min=-0.000, max=0.000
